# Memulai dengan Agen: Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/03-agen-latihan.ipynb)

Ini notebook **latihan** bab ini di Google Colab. Semua latihan (2 latihan) ada di satu notebook, dikerjakan berurutan dari atas ke bawah; setiap latihan diawali judul `Latihan N`. Solusi lengkap yang sudah terisi dan tinggal dijalankan ada di [notebook solusi](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari3/03-agen-solusi.ipynb).

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-agent-migas/blob/main/hari3/03-agen/03-agen.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Bab 3 - Latihan: Agen Analis Produksi
Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Agen dengan `create_agent()` yang menjawab pertanyaan multi-langkah dengan memadukan
pencarian dokumen, status sumur, perhitungan water cut, dan konversi satuan.

In [ ]:
# Langkah 1 - Menyiapkan model dan agen
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

from migas_shared.rag_tools import KonteksPengguna, cari_dokumen
from migas_shared.tampil import jejak_tool
from migas_shared.field_tools import TOOL_LAPANGAN

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

# TODO(B3.1): buat agen dengan create_agent: model, tools=[*TOOL_LAPANGAN, cari_dokumen], system_prompt analis produksi (angka dari tool, hitungan dari tool, prosedur dari cari_dokumen dengan menyebut ID dokumen), dan context_schema=KonteksPengguna
agent = None  # TODO: ganti dengan implementasi Anda

## Langkah 2 - Status sumur lalu perhitungan

In [ ]:
query = "Sumur SR-020 dan ML-012: berapa total produksi minyak keduanya, dan berapa water cut masing-masing?"
print(f"Pengguna: {query}")
# TODO(B3.2): panggil agent.invoke dengan HumanMessage dan context=KonteksPengguna(role="engineer")
response = None  # TODO: ganti dengan implementasi Anda
print("  Langkah  :")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Jawaban  : {response['messages'][-1].content}\n")

## Langkah 3 - Status sumur lalu konversi satuan

In [ ]:
query = "Berapa produksi minyak SR-020 dalam meter kubik per hari?"
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]}, context=KonteksPengguna(role="engineer"))
print("  Langkah  :")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Jawaban  : {response['messages'][-1].content}\n")

## Langkah 4 - Pencarian dokumen lalu ringkasan

In [ ]:
query = "Menurut SOP H2S yang berlaku, apa tindakan saat alarm tingkat 2 berbunyi di wellhead?"
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]}, context=KonteksPengguna(role="engineer"))
print("  Langkah  :")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Jawaban  : {response['messages'][-1].content}\n")

## Bab 3 - Latihan Bonus: Agen Perencana Pekerjaan Pulling
Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Agen dengan beberapa tool khusus yang butuh reasoning multi-langkah:
jadwal rig, perkiraan biaya, dan perkiraan produksi yang hilang.

In [ ]:
# Langkah 1 - Menyiapkan model
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from migas_shared.tampil import jejak_tool
from migas_shared.field_tools import cek_status_sumur

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Tool perencana pulling
Data fiktif untuk simulasi, lalu tiga tool: jadwal rig, biaya pulling, dan kehilangan produksi.

In [ ]:
# Data fiktif untuk simulasi
JADWAL_RIG = {"KN-033": 12, "SR-020": 5, "ML-012": 20}  # hari menunggu rig tersedia
BIAYA_PULLING_JUTA = {"SRP": 350, "ESP": 1200}  # juta rupiah per pekerjaan


# TODO(B3.3): buat tool cek_jadwal_rig(kode_sumur) yang mengembalikan berapa hari lagi rig tersedia dari JADWAL_RIG, atau pesan bila sumur tidak ada di jadwal
cek_jadwal_rig = None  # TODO: ganti dengan implementasi Anda


@tool
def estimasi_biaya_pulling(jenis_pompa: str) -> str:
    """Perkirakan biaya satu pekerjaan pulling berdasarkan jenis pompa (SRP atau ESP), dalam juta rupiah.

    Args:
        jenis_pompa: SRP atau ESP
    """
    biaya = BIAYA_PULLING_JUTA.get(jenis_pompa.upper())
    if biaya is None:
        return f"Jenis pompa {jenis_pompa} tidak dikenal. Pilihan: {', '.join(BIAYA_PULLING_JUTA)}"
    return f"Biaya pulling {jenis_pompa.upper()}: sekitar Rp {biaya} juta"


# TODO(B3.4): buat tool estimasi_kehilangan_produksi(minyak_bopd, hari) yang mengalikan keduanya dan menyebut hasil dalam bbl
estimasi_kehilangan_produksi = None  # TODO: ganti dengan implementasi Anda

## Langkah 3 - Membuat agen perencana

In [ ]:
# TODO(B3.5): buat agen dengan keempat tool dan system prompt perencana yang meminta rencana langkah demi langkah serta asumsi bila data tidak ada
agent = None  # TODO: ganti dengan implementasi Anda

## Langkah 4 - Rencana pulling satu sumur

In [ ]:
query = (
    "Rencanakan pulling sumur KN-033: kapan rig tersedia, berapa kira-kira biayanya, "
    "dan berapa produksi hilang selama menunggu jika produksi normalnya 45 bopd?"
)
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]})
print("  Langkah:")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Rencana: {response['messages'][-1].content}\n")

## Langkah 5 - Membandingkan dua sumur

In [ ]:
query = "Bandingkan KN-033 dan SR-020: sumur mana yang rig-nya lebih cepat tersedia dan berapa biaya pulling masing-masing?"
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]})
print("  Langkah:")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Rencana: {response['messages'][-1].content}\n")